# MAIN APPROACH

In [ ]:
import json
import time
import math
import re
import random
from bs4 import BeautifulSoup
import PIL.Image
import os
import glob
import io
import csv
import difflib
from google import genai
from google.genai import types
from google.colab import userdata
from google.api_core.exceptions import ServiceUnavailable, ResourceExhausted
import signal

# --- CONFIGURATION ---

# Output File for ALL results
MASTER_CSV_FILE = "MASTER_Correction_Dictionary.csv"

# BATCH SIZE
BATCH_SIZE = 100

# Setup Client
try:
    GEMINI_API_KEY = userdata.get('DSU_API_KEY')
    client = genai.Client(api_key=GEMINI_API_KEY)
    MODEL_ID = 'gemini-3-flash-preview'
    print("Connected to Gemini 3.0 Flash Preview (Thinking Level: Medium).")
except Exception as e:
    print(f"Setup Error: {e}")


# 3. AUTOMATED CROPPER
def get_batch_crop(image_path, batch_tags):
    img = PIL.Image.open(image_path)
    min_x, min_y = float('inf'), float('inf')
    max_x, max_y = float('-inf'), float('-inf')
    found_box = False

    for tag in batch_tags:
        title = tag.get('title', '')
        match = re.search(r'bbox (\d+) (\d+) (\d+) (\d+)', title)
        if match:
            found_box = True
            x0, y0, x1, y1 = map(int, match.groups())
            min_x = min(min_x, x0); min_y = min(min_y, y0)
            max_x = max(max_x, x1); max_y = max(max_y, y1)

    if not found_box: return None

    padding = 30
    crop_box = (
        max(0, min_x - padding), max(0, min_y - padding),
        min(img.width, max_x + padding), min(img.height, max_y + padding)
    )

    cropped_img = img.crop(crop_box)
    img_byte_arr = io.BytesIO()
    cropped_img.save(img_byte_arr, format='JPEG')
    return img_byte_arr.getvalue()

# 4. RETRY LOGIC
def generate_with_retry(contents, retries=5):
    wait = 5
    for attempt in range(retries):
        try:
            return client.models.generate_content(
                model=MODEL_ID,
                contents=contents,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0.0,
                    thinking_config=types.ThinkingConfig(
                       thinking_level="medium"
                    ),
                    # thinking_config=types.ThinkingConfig(
                    #    thinking_level=types.ThinkingLevel.MEDIUM
                    # ),
                    # thinking_config=types.ThinkingConfig(
                    #    thinking_budget=0
                    # ),
                )
            )
        except Exception as e:
            print(f"\n     ! Server busy (503). Waiting {wait}s...")
            time.sleep(wait)
            wait *= 2
    return None

# 5. POST-PROCESSING SAFETY FUNCTION (NEW)
# --- DEBUG SAFETY FUNCTION ---
def enforce_safety_rules_debug(original, ai_corrected):
    original = original.strip()
    ai_corrected = ai_corrected.strip()

    if original == ai_corrected:
        return original, "NO_CHANGE"

    # 1. HYPHEN RULE
    if original.endswith('-') and not ai_corrected.endswith('-'):
        if ai_corrected == original.rstrip('-'):
            return ai_corrected + "-", "HYPHEN_RESTORED"
        else:
            return original, "VETO_HYPHEN_LOSS"

    # 2. REVERSE LONG S RULE
    is_reverse_s_to_f = ('s' in original and 'f' in ai_corrected and 'f' not in original)
    is_valid_f_to_s = ('f' in original and 's' in ai_corrected and 'f' not in ai_corrected)

    if is_reverse_s_to_f and not is_valid_f_to_s:
        return original, "VETO_REVERSE_S_TO_F"

    # 3. ARCHAIC CHAR CLEANUP
    if 'ſ' in ai_corrected:
        ai_corrected = ai_corrected.replace('ſ', 's')
        return ai_corrected, "ACCEPTED_PLUS_CLEANUP"

    return ai_corrected, "ACCEPTED"

# --- 4. COMPARISON CORE FUNCTIONS ---

def extract_words(hocr_filepath):
    """Parses an hOCR file and returns a list of words."""
    try:
        with open(hocr_filepath, 'r', encoding='utf-8') as f:
            soup = BeautifulSoup(f, 'html.parser')
    except FileNotFoundError:
        print(f"🛑 Error: File not found at {hocr_filepath}.")
        return []

    words = []
    for word_span in soup.find_all('span', class_='ocrx_word'):
        word_text = word_span.get_text().strip()
        if word_text:
            words.append(word_text)
    return words

def get_changes_list(words1, words2, source_filename):
    """Aligns lists and returns ONLY the rows that have changes."""
    matcher = difflib.SequenceMatcher(None, words1, words2)
    changes = [] # List of [Source, Original, AI_Corrected, Type]

    for opcode, i1, i2, j1, j2 in matcher.get_opcodes():
        if opcode == 'equal':
            continue
        elif opcode == 'replace':
            max_len = max(i2 - i1, j2 - j1)
            for k in range(max_len):
                w1 = words1[i1 + k] if (i1 + k) < i2 else ""
                w2 = words2[j1 + k] if (j1 + k) < j2 else ""
                changes.append([source_filename, w1, w2, "REPLACED"])
        elif opcode == 'delete':
            for k in range(i2 - i1):
                changes.append([source_filename, words1[i1 + k], "", "DELETED"])
        elif opcode == 'insert':
            for k in range(j2 - j1):
                changes.append([source_filename, "", words2[j1 + k], "INSERTED"])
    return changes

def export_to_csv(values, filename):
    try:
        with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerows(values)
        print(f"\n✅ CSV Report generated: {filename}")
    except Exception as e:
        print(f"🛑 CSV Error: {e}")

# --- MAIN PIPELINE ---
class BatchTimeoutException(Exception):
    pass

def handle_batch_timeout(signum, frame):
    raise BatchTimeoutException()

def process_single_pair(hocr_file, img_file, csv_writer):
    print(f"\nProcessing: {hocr_file}...")
    output_hocr = hocr_file.replace(".hocr", "_AI_corrected.hocr")

    # --- PHASE 1: CORRECTION ---
    with open(hocr_file, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'html.parser')

    valid_words = []
    all_tags = soup.find_all('span', class_='ocrx_word')
    for idx, word in enumerate(all_tags):
        text = word.get_text().strip()
        if not text: continue
        if not word.has_attr('id'): word['id'] = f"gen_{idx}"
        valid_words.append(word)

    total = len(valid_words)
    print(f"Found {total} words. Processing in batches of {BATCH_SIZE}...")

    start_time = time.time()

    # STATISTICS COUNTERS
    stats = {"ai_proposals": 0, "vetoed": 0, "accepted": 0}

    for i in range(0, total, BATCH_SIZE):
        # if you want to not get stuck on a batch for more than 5 mins, uncomment this block
        # 1. Register the "timer" and set it to 300 seconds (5 mins)
        # signal.signal(signal.SIGALRM, handle_batch_timeout)
        # signal.alarm(300)

        batch_tags = valid_words[i : i + BATCH_SIZE]
        payload = {t['id']: t.get_text() for t in batch_tags}
        img_bytes = get_batch_crop(img_file, batch_tags)

        if not img_bytes:
            print(f"  Batch {i}: Skipped (No coords)")
            # if you want to not get stuck on a batch for more than 5 mins, uncomment this block
            # signal.alarm(0) # Disable alarm if skipping normally
            # continue

        print(f"\nBatch {i}-{i+len(batch_tags)} Processing...", end="")

        # Prompt
        prompt_text = f"""
# ROLE AND CONTEXT
You are an expert historical linguist and OCR-correction specialist for late 18th-century English and French texts (specifically the Quebec Gazette). Your task is to correct OCR errors in the provided JSON dictionary (id -> text) based strictly on the rules below.

# CORE RULES

1. THE "S" AND "F" OCR ERRORS (Highest Priority):
- The historical long 's' (ſ) is frequently misread by OCR.
- Convert any actual 'ſ' characters to a modern 's'.
- If a word contains an 'f' but makes no sense (e.g., "purofe", "Hofpital"), and replacing it with 's' creates a valid word ("purpose", "Hospital"), make the change.
- NEVER change a correct 's' into an 'f' or 'ſ'.

2. PRESERVATION OF VALID TEXT:
- If the word is already a valid English or French word, DO NOT touch it.
- Keep all archaic spellings exactly as they are (e.g., keep "publick", do not change to "public").
- Assume all digits and numbers are correct. Do not alter them.
- If a text segment ends with a hyphen (e.g., "situa-"), you MUST keep the hyphen.
- Do NOT add new accents/diacritics to French words if they are not already there.

3. LOOKALIKE CORRECTIONS:
- Fix common OCR shape-mix-ups (u/o, l/i/t, c/e) ONLY if the current text is nonsense and the correction forms a recognizable historical/modern word.

# EXAMPLES
Input:
{{"1": "ſituation", "2": "publick", "3": "purofe", "4": "1792", "5": "repu-", "6": "Hofpital", "7": "word", "8": "lntent"}}
Output:
{{"1": "situation", "2": "publick", "3": "purpose", "4": "1792", "5": "repu-", "6": "Hospital", "7": "word", "8": "Intent"}}

# OUTPUT FORMAT
- Return ONLY a raw JSON object.
- Do NOT wrap the output in ```json markdown code blocks.
- Do not include any greetings, explanations, or preambles.
- You MUST maintain the exact original IDs (keys) from the input.

# INPUT JSON:
{json.dumps(payload)}
        """

        # 3. Send to Gemini
        # The new SDK handles raw bytes with mime_type effectively in a list
        response = generate_with_retry([
            prompt_text,
            types.Part.from_bytes(data=img_bytes, mime_type='image/jpeg')
        ])

        # 4. ROBUST PARSING & POST-PROCESSING (New Safety Checks)
        if response:
            try:
                # finish_reason check is more robust in the new SDK
                if response.candidates[0].finish_reason not in ['STOP', 1]:
                    print(f" Failed (Finish Reason: {response.candidates[0].finish_reason})")
                    continue

                clean_text = response.text.strip()
                if clean_text.startswith("```"):
                    clean_text = re.sub(r"^```[a-zA-Z]*\n", "", clean_text)
                    clean_text = re.sub(r"\n```$", "", clean_text)

                corrections = json.loads(clean_text)

                batch_changes = 0

                # --- PROCESSING & DEBUG LOGGING ---
                for wid, new_text in corrections.items():
                    tag = soup.find('span', id=wid)
                    if tag:
                        original_text = tag.get_text().strip()
                        ai_corrected_text = new_text.strip()

                        # Check if AI actually proposed a change
                        if original_text != ai_corrected_text:
                            stats["ai_proposals"] += 1

                            # Run Safety Check
                            final_safe_text, status = enforce_safety_rules_debug(original_text, ai_corrected_text)

                            if status.startswith("VETO"):
                                stats["vetoed"] += 1
                                # DEBUG PRINT: See what is being rejected!
                                print(f"\n   [VETO] Orig: '{original_text}' | AI: '{ai_corrected_text}' | Reason: {status}")

                            elif status.startswith("ACCEPTED"):
                                stats["accepted"] += 1
                                tag.string = final_safe_text
                                batch_changes += 1
                                # Optional: Print accepted changes too if you want to see them
                                # print(f"\n   [OK] '{original_text}' -> '{final_safe_text}'")

                print(f" Done. ({batch_changes} changes applied)")

                # 2. IMPORTANT: Turn off the timer if the batch finished successfully
                signal.alarm(0)

            # if you want to not get stuck on a batch for more than 5 mins, uncomment this block
            # except BatchTimeoutException:
            #     # 3. This catches the 5-minute limit and moves to the next batch
            #     print(f"\nSkipping Batch {i}-{i+len(batch_tags)}. Took more than 5 mins.")
            #     signal.alarm(0) # Reset timer before continuing
            #     continue

            except Exception as e:
                # if you want to not get stuck on a batch for more than 5 mins, uncomment this block
                # Reset timer even if there is a random crash
                # signal.alarm(0)
                print(f" Error: {e}")
        else:
            print(" Failed (No Response)")

        time.sleep(2)

    with open(output_hocr, 'w', encoding='utf-8') as f:
        f.write(str(soup))

   # --- PHASE 2: COMPARISON & WRITING TO MASTER CSV ---
    words_orig = extract_words(hocr_file)
    words_new = extract_words(output_hocr)

    # Get changes and write to the shared CSV writer
    changes = get_changes_list(words_orig, words_new, hocr_file)
    if changes:
        csv_writer.writerows(changes)
        print(f"  ✅ Saved {len(changes)} corrections to master CSV.")

def run_batch_pipeline():
    start_time = time.time()

    MASTER_CSV = "MASTER_Correction_Dictionary.csv"
    print(f"--- 🚀 STARTING BATCH PIPELINE ---\nSaving to: {MASTER_CSV}")

    # Open the Master CSV once
    with open(MASTER_CSV, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["Source File", "Original Word", "AI Corrected Word", "Change Type"])

        # Find all HOCR files
        hocr_files = [f for f in glob.glob("*.hocr") if "_AI_corrected" not in f]

        if not hocr_files:
            print("❌ No .hocr files found.")
            return

        for hocr in hocr_files:
            # Auto-detect image file (jpg or png)
            base = os.path.splitext(hocr)[0]
            if os.path.exists(f"{base}.jpg"): img = f"{base}.jpg"
            elif os.path.exists(f"{base}.png"): img = f"{base}.png"
            else:
                print(f"⚠️ Skipping {hocr}: No image found.")
                continue

            try:
                process_single_pair(hocr, img, writer)
            except Exception as e:
                print(f"❌ Error on {hocr}: {e}")

    end_time = time.time()
    elapsed_time = end_time - start_time
    hours, rem = divmod(elapsed_time, 3600)
    minutes, seconds = divmod(rem, 60)

    print(f"\n🎉 BATCH PROCESSING COMPLETE!")
    print(f"⏱️ Total Execution Time: {int(hours)}h {int(minutes)}m {seconds:.2f}s")


if __name__ == "__main__":
    run_batch_pipeline()

# APPROACH 2 - Passing HOCR file content instead of text to gemini

In [ ]:
# import json
# import time
# import math
# import re
# import random
# from bs4 import BeautifulSoup
# import PIL.Image
# import os
# import glob
# import io
# import csv
# import difflib
# from google import genai
# from google.genai import types
# from google.colab import userdata
# from google.api_core.exceptions import ServiceUnavailable, ResourceExhausted

# # --- CONFIGURATION ---

# # Output File for ALL results
# MASTER_CSV_FILE = "MASTER_Correction_Dictionary.csv"

# # BATCH SIZE
# BATCH_SIZE = 100

# # Setup Client
# try:
#     GEMINI_API_KEY = userdata.get('DSU_API_KEY')
#     client = genai.Client(api_key=GEMINI_API_KEY)
#     MODEL_ID = 'gemini-3-flash-preview'
#     print("Connected to Gemini 3.0 Flash Preview.")
# except Exception as e:
#     print(f"Setup Error: {e}")


# # 3. AUTOMATED CROPPER
# def get_batch_crop(image_path, batch_tags):
#     img = PIL.Image.open(image_path)
#     min_x, min_y = float('inf'), float('inf')
#     max_x, max_y = float('-inf'), float('-inf')
#     found_box = False

#     for tag in batch_tags:
#         title = tag.get('title', '')
#         match = re.search(r'bbox (\d+) (\d+) (\d+) (\d+)', title)
#         if match:
#             found_box = True
#             x0, y0, x1, y1 = map(int, match.groups())
#             min_x = min(min_x, x0); min_y = min(min_y, y0)
#             max_x = max(max_x, x1); max_y = max(max_y, y1)

#     if not found_box: return None

#     padding = 30
#     crop_box = (
#         max(0, min_x - padding), max(0, min_y - padding),
#         min(img.width, max_x + padding), min(img.height, max_y + padding)
#     )

#     # # ADDED DEBUGGING PRINTS
#     # print(f"  DEBUG: min_x={min_x}, min_y={min_y}, max_x={max_x}, max_y={max_y}")
#     # print(f"  DEBUG: crop_box={crop_box}")
#     # if crop_box[0] >= crop_box[2] or crop_box[1] >= crop_box[3]:
#     #     print("  WARNING: Invalid crop_box detected (width or height is zero/negative).")

#     cropped_img = img.crop(crop_box)
#     img_byte_arr = io.BytesIO()
#     cropped_img.save(img_byte_arr, format='JPEG')
#     return img_byte_arr.getvalue()

# # 4. RETRY LOGIC
# def generate_with_retry(contents, retries=5):
#     wait = 5
#     for attempt in range(retries):
#         try:
#             return client.models.generate_content(
#                 model=MODEL_ID,
#                 contents=contents,
#                 config=types.GenerateContentConfig(
#                     response_mime_type="application/json",
#                     temperature=0.0,
#                     # thinking_config=types.ThinkingConfig(
#                     #   thinking_level=types.ThinkingLevel.LOW
#                     # ),
#                 )
#             )
#         except Exception as e:
#             print(f"\n     ! Server busy (503). Waiting {wait}s...")
#             time.sleep(wait)
#             wait *= 2
#     return None

# # 5. POST-PROCESSING SAFETY FUNCTION (NEW)
# # --- DEBUG SAFETY FUNCTION ---
# def enforce_safety_rules_debug(original, ai_corrected):
#     original = original.strip()
#     ai_corrected = ai_corrected.strip()

#     if original == ai_corrected:
#         return original, "NO_CHANGE"

#     # 1. HYPHEN RULE
#     if original.endswith('-') and not ai_corrected.endswith('-'):
#         if ai_corrected == original.rstrip('-'):
#             return ai_corrected + "-", "HYPHEN_RESTORED"
#         else:
#             return original, "VETO_HYPHEN_LOSS"

#     # 2. REVERSE LONG S RULE
#     is_reverse_s_to_f = ('s' in original and 'f' in ai_corrected and 'f' not in original)
#     is_valid_f_to_s = ('f' in original and 's' in ai_corrected and 'f' not in ai_corrected)

#     if is_reverse_s_to_f and not is_valid_f_to_s:
#         return original, "VETO_REVERSE_S_TO_F"

#     # 3. ARCHAIC CHAR CLEANUP
#     if 'ſ' in ai_corrected:
#         ai_corrected = ai_corrected.replace('ſ', 's')
#         return ai_corrected, "ACCEPTED_PLUS_CLEANUP"

#     return ai_corrected, "ACCEPTED"

# # --- 4. COMPARISON CORE FUNCTIONS ---

# def extract_words(hocr_filepath):
#     """Parses an hOCR file and returns a list of words."""
#     try:
#         with open(hocr_filepath, 'r', encoding='utf-8') as f:
#             soup = BeautifulSoup(f, 'html.parser')
#     except FileNotFoundError:
#         print(f"🛑 Error: File not found at {hocr_filepath}.")
#         return []

#     words = []
#     for word_span in soup.find_all('span', class_='ocrx_word'):
#         word_text = word_span.get_text().strip()
#         if word_text:
#             words.append(word_text)
#     return words

# def get_changes_list(words1, words2, source_filename):
#     """Aligns lists and returns ONLY the rows that have changes."""
#     matcher = difflib.SequenceMatcher(None, words1, words2)
#     changes = [] # List of [Source, Original, AI_Corrected, Type]

#     for opcode, i1, i2, j1, j2 in matcher.get_opcodes():
#         if opcode == 'equal':
#             continue
#         elif opcode == 'replace':
#             max_len = max(i2 - i1, j2 - j1)
#             for k in range(max_len):
#                 w1 = words1[i1 + k] if (i1 + k) < i2 else ""
#                 w2 = words2[j1 + k] if (j1 + k) < j2 else ""
#                 changes.append([source_filename, w1, w2, "REPLACED"])
#         elif opcode == 'delete':
#             for k in range(i2 - i1):
#                 changes.append([source_filename, words1[i1 + k], "", "DELETED"])
#         elif opcode == 'insert':
#             for k in range(j2 - j1):
#                 changes.append([source_filename, "", words2[j1 + k], "INSERTED"])
#     return changes

# def export_to_csv(values, filename):
#     try:
#         with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
#             writer = csv.writer(csvfile)
#             writer.writerows(values)
#         print(f"\n✅ CSV Report generated: {filename}")
#     except Exception as e:
#         print(f"🛑 CSV Error: {e}")

# # --- MAIN PIPELINE ---
# def process_single_pair(hocr_file, img_file, csv_writer):
#     print(f"\nProcessing: {hocr_file}...")
#     output_hocr = hocr_file.replace(".hocr", "_AI_corrected.hocr")

#     # --- PHASE 1: CORRECTION ---
#     with open(hocr_file, 'r', encoding='utf-8') as f:
#         soup = BeautifulSoup(f, 'html.parser')

#     valid_words = []
#     all_tags = soup.find_all('span', class_='ocrx_word')
#     for idx, word in enumerate(all_tags):
#         text = word.get_text().strip()
#         if not text: continue
#         if not word.has_attr('id'): word['id'] = f"gen_{idx}"
#         valid_words.append(word)

#     total = len(valid_words)
#     print(f"Found {total} words. Processing in batches of {BATCH_SIZE}...")

#     start_time = time.time()

#     # STATISTICS COUNTERS
#     stats = {"ai_proposals": 0, "vetoed": 0, "accepted": 0}

#     for i in range(0, total, BATCH_SIZE):
#         batch_tags = valid_words[i : i + BATCH_SIZE]

#         # --- NEW: Create a raw HTML string for the batch ---
#         batch_html = "".join([str(t) for t in batch_tags])

#         img_bytes = get_batch_crop(img_file, batch_tags)

#         if not img_bytes:
#             print(f"  Batch {i}: Skipped (No coords)")
#             continue

#         print(f"\nBatch {i}-{i+len(batch_tags)} Processing...", end="")

#         # Prompt
#         prompt_text = f"""
#         You are an expert historical archivist correcting OCR text from the 'Quebec Gazette' (late 18th century).
#         I have provided the raw hOCR HTML snippet and a cropped IMAGE of that text.

#         YOUR MISSION:
#         1. Read the provided hOCR HTML.
#         2. Compare the text within the <span> tags to the provided image.
#         3. Identify OCR errors.
#         4. Return a JSON object where the keys are the 'id' attributes from the HTML and the values are the CORRECTED text.

#         Compare the OCR to the image and fix OCR errors. You must prioritize the following Instructions.

#         --- 📢 CRUCIAL INSTRUCTIONS (SAFETY GUARDRAILS) ---

#         1. 🖼️ **IMAGE USE - VISUAL VETO (Anti-Hallucination):**
#            - **Trust Text Over Image for Valid Words:** If the input text is *already* a valid, correctly spelled word (e.g., 'situation', 'stated'), **DO NOT CHANGE IT** based on the image. The image is often blurry or misleading (e.g., making 's' look like 'f').
#            - **Rule:** Never change a valid word (e.g., 'situation') into a nonsense word (e.g., 'fiftation') just because the image looks that way.

#         2. 🔢 **NUMBER PRESERVATION (Strict):**
#            - **NEVER** change numbers or digits (e.g., '200,000' must NOT become '300,000').
#            - Assume the OCR text is correct for all digits. Do not "correct" numbers based on the image.

#         3. 🚫 **NO ARCHAIC OUTPUT (The "s" Rule):**
#            - While the image may show the historical Long S (ſ), your output MUST use the standard modern 's'.
#            - **NEVER** output the character 'ſ'. Always convert it to 's'.
#            - **NEVER** change a standard 's' back to 'ſ' or 'f'.

#         4. ⛔ **STRICT DIACRITIC RULE:**
#            - **DO NOT ADD DIACRITICS**: If the original OCR text does not have an accent (e.g., 'Majefté', 'Été', 'à'), **DO NOT ADD ONE**.
#            - **Authenticity First**: You must preserve the original text's lack of accents.
#            - **Exception**: Only change a diacritic if the 'original OCR' had it wrong.

#         --- 🔎 KNOWN ERROR CHECKLIST ---

#         1. **The "Long S" Confusion & Real Word Heuristic:**
#            - **Error:** The OCR frequently reads the historical "Long S" (ſ) as 'f' (e.g., 'purofe' for 'purpose').
#            - **Rule (Real Word Goal):** If substituting the misread **'f' with 's'** (or the historical long 'ſ') results in a **valid, recognizable word** (in English or French, consistent with the document's language), **then perform the correction.**
#            - **Target:** 'f' -> 's' is the most important fix.

#         2. **Hyphenation Preservation Rule:**
#            - **Rule:** If a word segment ends with a **hyphen (`-`)** (indicating a word split by a line break, e.g., 'nati-' continuing as 'on'), you **MUST preserve the hyphen** in the output. DO NOT delete the hyphen to complete the word. This maintains the original typographical characteristic.

#         3. **Visually Lookalikes:**
#            - Match for 'u' misread as 'o' (e.g., 'jout' -> 'just').
#            - Match for 'l' misread as 'i' or 't'.
#            - **Rule:** If a word looks like nonsense (e.g., 'ifland'), check if a lookalike swap fixes it.

#         --- GENERAL CONSTRAINTS ---
#         1. **Mandatory Word-by-Word Review:** You MUST process the input text word-by-word, applying all rules from the "KNOWN ERROR CHECKLIST" before generating the final output.
#         2. **No Modernization:** Do not update spelling (e.g., keep 'colour', 'publick').
#         3. **Internal Knowledge Only:** Do NOT use external internet search. However, **DO** use your internal linguistic knowledge of English and French to validate if a word is real.
#         4. **Grammar:** Do NOT correct grammar, punctuation, or sentence structure.
#         5. **Output Format:**
#            - Return a JSON object with the EXACT same keys.
#            - Return raw JSON only (no markdown, no preamble).

#         Input hOCR HTML:
#         {batch_html}
#         """

#         # 3. Send to Gemini
#         # The new SDK handles raw bytes with mime_type effectively in a list
#         response = generate_with_retry([
#             prompt_text,
#             types.Part.from_bytes(data=img_bytes, mime_type='image/jpeg')
#         ])

#         # 4. ROBUST PARSING & POST-PROCESSING (New Safety Checks)
#         if response:
#             try:
#                 # finish_reason check is more robust in the new SDK
#                 if response.candidates[0].finish_reason not in ['STOP', 1]:
#                     print(f" Failed (Finish Reason: {response.candidates[0].finish_reason})")
#                     continue

#                 clean_text = response.text.strip()
#                 if clean_text.startswith("```"):
#                     clean_text = re.sub(r"^```[a-zA-Z]*\n", "", clean_text)
#                     clean_text = re.sub(r"\n```$", "", clean_text)

#                 corrections = json.loads(clean_text)

#                 batch_changes = 0

#                 # --- PROCESSING & DEBUG LOGGING ---
#                 for wid, new_text in corrections.items():
#                     tag = soup.find('span', id=wid)
#                     if tag:
#                         original_text = tag.get_text().strip()
#                         ai_corrected_text = new_text.strip()

#                         # Check if AI actually proposed a change
#                         if original_text != ai_corrected_text:
#                             stats["ai_proposals"] += 1

#                             # Run Safety Check
#                             final_safe_text, status = enforce_safety_rules_debug(original_text, ai_corrected_text)

#                             if status.startswith("VETO"):
#                                 stats["vetoed"] += 1
#                                 # DEBUG PRINT: See what is being rejected!
#                                 print(f"\n   [VETO] Orig: '{original_text}' | AI: '{ai_corrected_text}' | Reason: {status}")

#                             elif status.startswith("ACCEPTED"):
#                                 stats["accepted"] += 1
#                                 tag.string = final_safe_text
#                                 batch_changes += 1
#                                 # Optional: Print accepted changes too if you want to see them
#                                 # print(f"\n   [OK] '{original_text}' -> '{final_safe_text}'")

#                 print(f" Done. ({batch_changes} changes applied)")

#             except Exception as e:
#                 print(f" Error: {e}")
#         else:
#             print(" Failed (No Response)")

#         time.sleep(2)

#     with open(output_hocr, 'w', encoding='utf-8') as f:
#         f.write(str(soup))

#    # --- PHASE 2: COMPARISON & WRITING TO MASTER CSV ---
#     words_orig = extract_words(hocr_file)
#     words_new = extract_words(output_hocr)

#     # Get changes and write to the shared CSV writer
#     changes = get_changes_list(words_orig, words_new, hocr_file)
#     if changes:
#         csv_writer.writerows(changes)
#         print(f"  ✅ Saved {len(changes)} corrections to master CSV.")

# def run_batch_pipeline():
#     start_time = time.time()

#     MASTER_CSV = "MASTER_Correction_Dictionary.csv"
#     print(f"--- 🚀 STARTING BATCH PIPELINE ---\nSaving to: {MASTER_CSV}")

#     # Open the Master CSV once
#     with open(MASTER_CSV, 'w', newline='', encoding='utf-8') as csvfile:
#         writer = csv.writer(csvfile)
#         writer.writerow(["Source File", "Original Word", "AI Corrected Word", "Change Type"])

#         # Find all HOCR files
#         hocr_files = [f for f in glob.glob("*.hocr") if "_AI_corrected" not in f]

#         if not hocr_files:
#             print("❌ No .hocr files found.")
#             return

#         for hocr in hocr_files:
#             # Auto-detect image file (jpg or png)
#             base = os.path.splitext(hocr)[0]
#             if os.path.exists(f"{base}.jpg"): img = f"{base}.jpg"
#             elif os.path.exists(f"{base}.png"): img = f"{base}.png"
#             else:
#                 print(f"⚠️ Skipping {hocr}: No image found.")
#                 continue

#             try:
#                 process_single_pair(hocr, img, writer)
#             except Exception as e:
#                 print(f"❌ Error on {hocr}: {e}")

#     end_time = time.time()
#     elapsed_time = end_time - start_time
#     hours, rem = divmod(elapsed_time, 3600)
#     minutes, seconds = divmod(rem, 60)

#     print(f"\n🎉 BATCH PROCESSING COMPLETE!")
#     print(f"⏱️ Total Execution Time: {int(hours)}h {int(minutes)}m {seconds:.2f}s")


# if __name__ == "__main__":
#     run_batch_pipeline()

# APPROACH 3 - Passing HOCR file as attachment instead of text to gemini

In [ ]:
# import json
# import time
# import math
# import re
# import random
# from bs4 import BeautifulSoup
# import PIL.Image
# import os
# import glob
# import io
# import csv
# import difflib
# from google import genai
# from google.genai import types
# from google.colab import userdata
# from google.api_core.exceptions import ServiceUnavailable, ResourceExhausted

# # --- CONFIGURATION ---

# # Output File for ALL results
# MASTER_CSV_FILE = "MASTER_Correction_Dictionary.csv"

# # Setup Client
# try:
#     GEMINI_API_KEY = userdata.get('DSU_API_KEY')
#     client = genai.Client(api_key=GEMINI_API_KEY)
#     MODEL_ID = 'gemini-3-pro-preview'
#     print("Connected to Gemini 3.0 Pro Preview.")
# except Exception as e:
#     print(f"Setup Error: {e}")


# # 4. RETRY LOGIC
# def generate_with_retry(contents, retries=5):
#     wait = 5
#     for attempt in range(retries):
#         try:
#             return client.models.generate_content(
#                 model=MODEL_ID,
#                 contents=contents,
#                 config=types.GenerateContentConfig(
#                     response_mime_type="application/json",
#                     temperature=0.0,
#                     thinking_config=types.ThinkingConfig(
#                       thinking_level=types.ThinkingLevel.LOW
#                     ),
#                 )
#             )
#         except Exception as e:
#             print(f"\n     ! Server busy (503). Waiting {wait}s...")
#             time.sleep(wait)
#             wait *= 2
#     return None

# # 5. POST-PROCESSING SAFETY FUNCTION (NEW)
# # --- DEBUG SAFETY FUNCTION ---
# def enforce_safety_rules_debug(original, ai_corrected):
#     original = original.strip()
#     ai_corrected = ai_corrected.strip()

#     if original == ai_corrected:
#         return original, "NO_CHANGE"

#     # 1. HYPHEN RULE
#     if original.endswith('-') and not ai_corrected.endswith('-'):
#         if ai_corrected == original.rstrip('-'):
#             return ai_corrected + "-", "HYPHEN_RESTORED"
#         else:
#             return original, "VETO_HYPHEN_LOSS"

#     # 2. REVERSE LONG S RULE
#     is_reverse_s_to_f = ('s' in original and 'f' in ai_corrected and 'f' not in original)
#     is_valid_f_to_s = ('f' in original and 's' in ai_corrected and 'f' not in ai_corrected)

#     if is_reverse_s_to_f and not is_valid_f_to_s:
#         return original, "VETO_REVERSE_S_TO_F"

#     # 3. ARCHAIC CHAR CLEANUP
#     if 'ſ' in ai_corrected:
#         ai_corrected = ai_corrected.replace('ſ', 's')
#         return ai_corrected, "ACCEPTED_PLUS_CLEANUP"

#     return ai_corrected, "ACCEPTED"

# # --- 4. COMPARISON CORE FUNCTIONS ---

# def extract_words(hocr_filepath):
#     """Parses an hOCR file and returns a list of words."""
#     try:
#         with open(hocr_filepath, 'r', encoding='utf-8') as f:
#             soup = BeautifulSoup(f, 'html.parser')
#     except FileNotFoundError:
#         print(f"🛑 Error: File not found at {hocr_filepath}.")
#         return []

#     words = []
#     for word_span in soup.find_all('span', class_='ocrx_word'):
#         word_text = word_span.get_text().strip()
#         if word_text:
#             words.append(word_text)
#     return words

# def get_changes_list(words1, words2, source_filename):
#     """Aligns lists and returns ONLY the rows that have changes."""
#     matcher = difflib.SequenceMatcher(None, words1, words2)
#     changes = [] # List of [Source, Original, AI_Corrected, Type]

#     for opcode, i1, i2, j1, j2 in matcher.get_opcodes():
#         if opcode == 'equal':
#             continue
#         elif opcode == 'replace':
#             max_len = max(i2 - i1, j2 - j1)
#             for k in range(max_len):
#                 w1 = words1[i1 + k] if (i1 + k) < i2 else ""
#                 w2 = words2[j1 + k] if (j1 + k) < j2 else ""
#                 changes.append([source_filename, w1, w2, "REPLACED"])
#         elif opcode == 'delete':
#             for k in range(i2 - i1):
#                 changes.append([source_filename, words1[i1 + k], "", "DELETED"])
#         elif opcode == 'insert':
#             for k in range(j2 - j1):
#                 changes.append([source_filename, "", words2[j1 + k], "INSERTED"])
#     return changes

# def export_to_csv(values, filename):
#     try:
#         with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
#             writer = csv.writer(csvfile)
#             writer.writerows(values)
#         print(f"\n✅ CSV Report generated: {filename}")
#     except Exception as e:
#         print(f"🛑 CSV Error: {e}")

# # --- MAIN PIPELINE ---
# def process_single_pair(hocr_file, img_file, csv_writer):
#     print(f"\n--- Testing Single Call (HOCR as File) ---")
#     output_hocr = hocr_file.replace(".hocr", "_AI_corrected.hocr")

#     # 1. Read Image Bytes
#     with open(img_file, 'rb') as f:
#         img_bytes = f.read()

#     # 2. Read HOCR Bytes
#     with open(hocr_file, 'rb') as f:
#         hocr_bytes = f.read()

#     # 3. Load soup for local application later
#     soup = BeautifulSoup(hocr_bytes, 'html.parser')

#     # Prompt
#     prompt_text = f"""
#     ### ROLE & CONTEXT
#     You are an expert at correcting HOCR content in English and French bilingual from the from the 'Quebec Gazette' (late 18th century).

#     ### INPUT
#     Two files are uploaded.  One is an image file that is one page of the Quebec Gazette newspaper.  Another is a snippet corresponding to its HOCR file.

#     ### INSTRUCTIONS
#     Review the entire HOCR file.  Correct the HOCR words using the following rules:
#     - Words can be identified by values within class="ocrx_word".  Only correct words.  Do NOT correct the elements or attrributes of the HOCR.
#     - **Trust Text Over Image for Valid Words:** If the input text is *already* a valid, correctly spelled word (e.g., 'situation', 'stated'), **DO NOT CHANGE IT** based on the image. The image is often blurry or misleading (e.g., making 's' look like 'f').
#     - **NEVER** change numbers or digits (e.g., '200,000' must NOT become '300,000').
#     - While the image may show the historical Long S (ſ), your output MUST use the standard modern 's'.
#     - **NEVER** output the character 'ſ'. Always convert it to 's'.
#     - **NEVER** change a standard 's' back to 'ſ' or 'f'.
#     - **DO NOT ADD DIACRITICS**: If the original OCR text does not have an accent (e.g., 'Majefté', 'Été', 'à'), **DO NOT ADD ONE**.
#     - **Rule:** If a word segment ends with a **hyphen (`-`)** (indicating a word split by a line break, e.g., 'nati-' continuing as 'on'), you **MUST preserve the hyphen** in the output. DO NOT delete the hyphen to complete the word. This maintains the original typographical characteristic.
#     - **Visually Lookalikes:**
#         - Match for 'u' misread as 'o' (e.g., 'jout' -> 'just').
#         - Match for 'l' misread as 'i' or 't'.
#         - **Rule:** If a word looks like nonsense (e.g., 'ifland'), check if a lookalike swap fixes it.

#     ### OUTPUT
#     Following the above rules, correct the HOCR.
#     **Output Format:**
#         - Return a JSON object only (no markdown, no preamble).
#     """

#     # 5. API Call with Multiple Parts
#     print("Sending Image and HOCR file to Gemini...")
#     response = generate_with_retry([
#         prompt_text,
#         types.Part.from_bytes(data=img_bytes, mime_type='image/jpeg'),
#         types.Part.from_bytes(data=hocr_bytes, mime_type='text/html') # Passing HOCR as its own part
#     ])

#     # 6. Apply Corrections
#     if response:
#         try:
#             clean_text = response.text.strip()
#             if clean_text.startswith("```"):
#                 clean_text = re.sub(r"^```[a-zA-Z]*\n", "", clean_text)
#                 clean_text = re.sub(r"\n```$", "", clean_text)

#             corrections = json.loads(clean_text)
#             changes_applied = 0

#             for wid, new_text in corrections.items():
#                 tag = soup.find('span', id=wid)
#                 if tag:
#                     original_text = tag.get_text().strip()
#                     final_safe_text, status = enforce_safety_rules_debug(original_text, new_text)

#                     if status.startswith("ACCEPTED"):
#                         tag.string = final_safe_text
#                         changes_applied += 1

#             print(f"Done. Applied {changes_applied} changes.")

#             with open(output_hocr, 'w', encoding='utf-8') as f:
#                 f.write(str(soup))

#             # Record for CSV
#             words_orig = extract_words(hocr_file)
#             words_new = extract_words(output_hocr)
#             changes = get_changes_list(words_orig, words_new, hocr_file)
#             if changes:
#                 csv_writer.writerows(changes)

#         except Exception as e:
#             print(f"Parsing/Correction Error: {e}")
#     else:
#         print("Failed: No response.")


# # updated main execution (only one file and remove the batch logic)
# if __name__ == "__main__":
#     TEST_HOCR = "para1.hocr.txt"
#     TEST_IMAGE = "005_02_0.jpg"
#     MASTER_CSV = "SINGLE_TEST_RESULTS.csv"

#     with open(MASTER_CSV, 'w', newline='', encoding='utf-8') as csvfile:
#         writer = csv.writer(csvfile)
#         writer.writerow(["Source File", "Original Word", "AI Corrected Word", "Change Type"])

#         if os.path.exists(TEST_HOCR) and os.path.exists(TEST_IMAGE):
#             process_single_pair(TEST_HOCR, TEST_IMAGE, writer)
#         else:
#             print("Error: Ensure your test HOCR and JPG files exist in the directory.")